# Tái hiện NVAUTO (bài 22, ISLES 2022) trên Kaggle

Auto3DSeg SegResNet, DWI + ADC, patch 192×192×128, Dice + Focal có deep supervision, AdamW 2e-4 cosine, 1000 epoch, 5-fold.

**Trước khi chạy:** Accelerator = *GPU T4 x2*, Internet = *On* (để cài MONAI), và thêm 2 input:
1. Dataset ISLES-2022 của nhóm (thư mục có `sub-strokecase*/` và `derivatives/`).
2. Dataset chứa thư mục `nvauto_repro` (code này).

Từ phiên thứ 2 trở đi: thêm output của phiên trước làm input, rồi đặt `RESUME_FROM` ở ô cấu hình. Xem `README.md`.

In [ ]:
# ===== Cấu hình: chỉ cần sửa ô này =====
FOLDS = "0 1 2 3 4"   # nên chạy "0" trước để đo tốc độ
EPOCHS = 1000          # bài 22: 1000
HOURS = 11.3           # tự dừng an toàn trước giới hạn 12 giờ của Kaggle
RESUME_FROM = ""       # vd "/kaggle/input/nvauto-run-1" (output phiên trước); để trống ở phiên đầu
DATA_ROOT = "auto"     # hoặc đường dẫn tới thư mục chứa sub-strokecase*/ và derivatives/

In [ ]:
!pip install -q "monai==1.6.1" nibabel fire
import glob, shutil, subprocess
src = sorted(glob.glob("/kaggle/input/**/nvauto_repro/run_nvauto.py", recursive=True))
assert src, "Không tìm thấy nvauto_repro trong /kaggle/input"
shutil.copytree(src[0].rsplit("/", 1)[0], "/kaggle/working/code", dirs_exist_ok=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

In [ ]:
resume = f"--resume-from {RESUME_FROM}" if RESUME_FROM else ""
!python /kaggle/working/code/run_nvauto.py train --folds {FOLDS} --epochs {EPOCHS} --hours {HOURS} --data-root {DATA_ROOT} {resume}

In [ ]:
# Chỉ chạy khi mọi fold đã "done": Dice từng ca ở độ phân giải gốc, trung bình + trung vị
# !python /kaggle/working/code/run_nvauto.py evaluate --folds {FOLDS} --epochs {EPOCHS} --data-root {DATA_ROOT} {resume}